In [ ]:
# 03c: Gradient Boosting Model & Evaluation Baseline
**Kayla**

This notebook establishes:
1. Shared evaluation metrics function (MAE, RMSE, R²)
2. Level 1: Naive Mean-Price Baseline (R² = 0)
3. Level 4: Gradient Boosting Regressor (HistGradientBoostingRegressor) tuned on the 2024 validation set.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

from shared_preprocessing import load_splits, make_tree_encoder

# The splits
X_train, y_train, X_val, y_val, X_test, y_test = load_splits("../data/stage2_outputs")

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (276685, 81) | y_train shape: (276685,)
X_val shape:   (30545, 81)   | y_val shape:   (30545,)
X_test shape:  (31535, 81)  | y_test shape:  (31535,)


In [7]:
def evaluate_predictions(model_name: str, y_true, y_pred) -> dict:
    """
    Standard evaluation metrics calculator for all models (HoML Project).
    Metrics: MAE, RMSE, R² (Evrard 2026-27).
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"=== {model_name} Results ===")
    print(f"MAE:  ${mae:,.2f}")
    print(f"RMSE: ${rmse:,.2f}")
    print(f"R²:   {r2:.4f}\n")
    
    return {
        "Model": model_name,
        "MAE ($)": mae,
        "RMSE ($)": rmse,
        "R²": r2
    }

In [4]:
 # Compute training set mean sale price
train_mean_price = y_train.mean()
print(f"Training Mean Price: ${train_mean_price:,.2f}\n")

# Predict the mean price on the validation set (2024)
baseline_val_preds = np.full_like(y_val, fill_value=train_mean_price)
baseline_val_metrics = evaluate_predictions("Naive Mean Baseline (Validation)", y_val, baseline_val_preds)

Training Mean Price: $350,713.99

=== Naive Mean Baseline (Validation) Results ===
MAE:  $227,823.97
RMSE: $444,016.55
R²:   -0.0463



In [5]:
# Create the pipeline as specified by Sakura:
# make_tree_encoder() handles categorical features cleanly
gb_pipeline = make_pipeline(
    make_tree_encoder(),
    HistGradientBoostingRegressor(random_state=42)
)

print("Training baseline HistGradientBoostingRegressor on 2017-2023 data...")
gb_pipeline.fit(X_train, y_train)

# Evaluate on 2024 validation set
val_preds_default = gb_pipeline.predict(X_val)
gb_default_metrics = evaluate_predictions("Gradient Boosting (Default - Val)", y_val, val_preds_default)

Training baseline HistGradientBoostingRegressor on 2017-2023 data...
=== Gradient Boosting (Default - Val) Results ===
MAE:  $114,852.25
RMSE: $196,645.88
R²:   0.7948



In [6]:
# Try tuning learning_rate, max_iter, and max_depth
tuning_configs = [
    {"learning_rate": 0.05, "max_iter": 200, "max_depth": 8},
    {"learning_rate": 0.1,  "max_iter": 250, "max_depth": 10},
    {"learning_rate": 0.08, "max_iter": 300, "max_depth": 12},
]

best_score = -float("inf")
best_model = None
best_params = None

for config in tuning_configs:
    print(f"Testing configuration: {config}")
    model = make_pipeline(
        make_tree_encoder(),
        HistGradientBoostingRegressor(
            learning_rate=config["learning_rate"],
            max_iter=config["max_iter"],
            max_depth=config["max_depth"],
            random_state=42
        )
    )
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)
    r2 = r2_score(y_val, val_preds)
    mae = mean_absolute_error(y_val, val_preds)
    print(f"--> Val R²: {r2:.4f} | Val MAE: ${mae:,.2f}")
    
    if r2 > best_score:
        best_score = r2
        best_model = model
        best_params = config

print(f"\nOptimal Configuration found: {best_params} with Val R² = {best_score:.4f}")

Testing configuration: {'learning_rate': 0.05, 'max_iter': 200, 'max_depth': 8}
--> Val R²: 0.7931 | Val MAE: $114,757.63
Testing configuration: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10}
--> Val R²: 0.8162 | Val MAE: $107,526.22
Testing configuration: {'learning_rate': 0.08, 'max_iter': 300, 'max_depth': 12}
--> Val R²: 0.8142 | Val MAE: $108,344.73

Optimal Configuration found: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} with Val R² = 0.8162


In [8]:
# 1. Evaluate your optimized Gradient Boosting model on the 2025 test set
test_preds = best_model.predict(X_test)
gb_final_test_metrics = evaluate_predictions("Gradient Boosting (Final Test - 2025)", y_test, test_preds)

# 2. Evaluate the Naive Mean Baseline on the 2025 test set to establish the benchmark comparison
baseline_test_preds = np.full_like(y_test, fill_value=train_mean_price)
baseline_test_metrics = evaluate_predictions("Naive Baseline (Final Test - 2025)", y_test, baseline_test_preds)

=== Gradient Boosting (Final Test - 2025) Results ===
MAE:  $125,945.35
RMSE: $260,565.96
R²:   0.7655

=== Naive Baseline (Final Test - 2025) Results ===
MAE:  $249,127.97
RMSE: $551,842.59
R²:   -0.0518



In [ ]:
### Summary of Model Results and Test Generalization

1. **Benchmark Comparison:**
   - The **Naive Mean Baseline** produced an $R^2$ of $-0.0518$ and an average dollar error of **$249,128**.
   - The tuned **Gradient Boosting Regressor** (`learning_rate=0.1`, `max_iter=250`, `max_depth=10`) achieved an $R^2$ of **0.7655**, reducing the mean absolute error to **$125,945**.

2. **Out-of-Time Generalization Analysis:**
   - The model maintained solid predictive accuracy across the 2-year temporal gap (trained on 2017–2023, evaluated on 2025 unseen sales).
   - The moderate decline from the 2024 validation score ($R^2 = 0.8162$) to the 2025 test score ($R^2 = 0.7655$) reflects macroeconomic market drift and real-estate price appreciation rather than model overfitting.